# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [78]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings('ignore')

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [79]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, df, y=None):
        return self

    def transform(self, df):
        df = df.copy()
        df["timestamp"] = df["timestamp"].astype("datetime64[ns]")
        df["hour"]  = df["timestamp"].dt.hour
        df["dayofweek"] = df["timestamp"].dt.weekday
        df.drop(columns="timestamp", inplace=True)
        return df

In [80]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target):
        self.target = target
        self.cat_cols = []

    def fit(self, df, y=None):
        df = df.copy()
        self.cat_cols = []
        for col in df.columns:
            if df[col].dtype in ["object", "category", "str"] and col != self.target:
                self.cat_cols.append(col)
        if self.cat_cols:  
            self.encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  
            self.encoder.fit(df[self.cat_cols])
        return self
    
    def transform(self, df):
        df = df.copy()
        target_col = df[self.target]
        
        if self.cat_cols: 
            encoded = self.encoder.transform(df[self.cat_cols])
            encoded_df = pd.DataFrame(
                encoded, 
                columns=self.encoder.get_feature_names_out(self.cat_cols), 
                index=df.index
            )
            df.drop(columns=self.cat_cols, inplace=True)
            df = pd.concat([df, encoded_df], axis=1)
        
        df.drop(columns=self.target, inplace=True)
        self.target_col = target_col
        return df

In [81]:
class TrainValidationTest():
    def transform(self, X, y):
        X = X.copy()
        y = y.copy()
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
        X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [82]:
class ModelSelection():
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.res = []
    
    def choose(self, X_train, y_train, X_valid, y_valid):
        for i, grid in tqdm(enumerate(self.grids), total=len(self.grids)):
            name = self.grid_dict[i]
            print(f"Estimator: {name}")
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_
            y_pred = best_model.predict(X_valid)
            acc = accuracy_score(y_valid, y_pred)
            print(f"Best params: {grid.best_params_}")
            print(f"Best training accuracy: {grid.best_score_:.3f}")
            print(f"validation set accuracy score for best params: {acc:.3f}\n")
            self.res.append({"model": name, "params": grid.best_params_, "valid_score": acc})
        best_i = np.argmax([row["valid_score"] for row in self.res])
        self.best_model = self.grids[best_i].best_estimator_
        return self.res[best_i]["model"]
    
    def best_results(self):
        return pd.DataFrame(self.res)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [83]:
class Finalize():
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {acc}")
        return acc

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print("Model was successfully saved")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [84]:
df = pd.read_csv("../../datasets/checker_submits.csv")

In [85]:
preprocessing = Pipeline([("feature_extractor", FeatureExtractor()), ("onehot_encoder", MyOneHotEncoder("dayofweek"))])

In [86]:
data = preprocessing.fit_transform(df)

In [87]:
data.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [88]:
X = data
y = preprocessing.named_steps["onehot_encoder"].target_col

In [89]:
y.head()

0    4
1    4
2    4
3    4
4    4
Name: dayofweek, dtype: int32

In [90]:
splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.transform(X, y)

In [91]:
svc = SVC()
tree = DecisionTreeClassifier()
forest = RandomForestClassifier()

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'),
                'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'],
                'class_weight':('balanced', None), 'random_state':[21],
                'probability':[True]}]
gs_svc = GridSearchCV(estimator=svc, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=-1)

tree_params = [{"max_depth": list(range(1, 50)),
                "class_weight": ["balanced", None],
                "criterion": ["entropy", "gini"],
                'random_state':[21]}]
gs_tree  = GridSearchCV(estimator=tree, param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=-1)

forest_params = [{"n_estimators": [5, 10, 50, 100],
                  "max_depth": list(range(1, 50)),
                  "class_weight": ["balanced", None],
                  "random_state":[21]}]
gs_forest = GridSearchCV(estimator=forest, param_grid=forest_params, scoring='accuracy', cv=2, n_jobs=-1)

grids = [gs_svc, gs_tree, gs_forest]
grid_dict = {0: "svc", 1: "DecisionTree", 2: "RandomForest"}


In [92]:
selector = ModelSelection(grids, grid_dict)
best_model_name = selector.choose(X_train, y_train, X_valid, y_valid)
best_model_name

  0%|          | 0/3 [00:00<?, ?it/s]

Estimator: svc
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
validation set accuracy score for best params: 0.878

Estimator: DecisionTree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
validation set accuracy score for best params: 0.863

Estimator: RandomForest
Best params: {'class_weight': None, 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.854
validation set accuracy score for best params: 0.904



'RandomForest'

In [93]:
best_params_df = selector.best_results()
best_params_df

,model,params,valid_score
0,svc,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.877778
1,DecisionTree,"{'class_weight': 'balanced', 'criterion': 'gin...",0.862963
2,RandomForest,"{'class_weight': None, 'max_depth': 22, 'n_est...",0.903704


In [94]:
final = Finalize(selector.best_model)
acc = final.final_score(X_train, y_train, X_test, y_test)
final.save_model(f"{best_model_name}_{acc}.sav")

Accuracy of the final model is 0.9112426035502958
Model was successfully saved


In [95]:
loaded_model = joblib.load(f"{best_model_name}_{acc}.sav")

In [96]:
final2 = Finalize(loaded_model)
acc = final.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.9112426035502958


In [97]:
preprocessing2 = Pipeline([("feature_extractor", FeatureExtractor()), ("onehot_encoder", MyOneHotEncoder("labname"))])

In [98]:
data2 = preprocessing2.fit_transform(df)
data2

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,uid_user_27,uid_user_28,uid_user_29,uid_user_3,uid_user_30,uid_user_31,uid_user_4,uid_user_6,uid_user_7,uid_user_8
0,1,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,3,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,4,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,5,5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1682,6,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1683,7,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1684,8,20,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [99]:
data2.columns

Index(['numTrials', 'hour', 'dayofweek', 'uid_user_0', 'uid_user_1',
       'uid_user_10', 'uid_user_11', 'uid_user_12', 'uid_user_13',
       'uid_user_14', 'uid_user_15', 'uid_user_16', 'uid_user_17',
       'uid_user_18', 'uid_user_19', 'uid_user_2', 'uid_user_20',
       'uid_user_21', 'uid_user_22', 'uid_user_23', 'uid_user_24',
       'uid_user_25', 'uid_user_26', 'uid_user_27', 'uid_user_28',
       'uid_user_29', 'uid_user_3', 'uid_user_30', 'uid_user_31', 'uid_user_4',
       'uid_user_6', 'uid_user_7', 'uid_user_8'],
      dtype='str')

In [100]:
y2 = preprocessing2.named_steps["onehot_encoder"].target_col
y2

0       project1
1       project1
2       project1
3       project1
4       project1
          ...   
1681     laba06s
1682     laba06s
1683     laba06s
1684     laba06s
1685     laba06s
Name: labname, Length: 1686, dtype: str